# 📊 台股 AI 投資戰情室 v4.0 Pro — Colab 啟動版

**執行順序（依序 Shift+Enter）：**

| Cell | 說明 | 時間 |
|------|------|------|
| **Cell 1** | 🔑 填入 API 金鑰 | 30 秒 |
| **Cell 2** | 📦 安裝套件 + 下載最新程式碼 | 3-5 分鐘 |
| **Cell 3** | 🔧 環境修復（uvloop / asyncio）| 10 秒 |
| **Cell 4** | 🔍 診斷測試（選擇性執行）| 1 分鐘 |
| **Cell 5** | 🚀 啟動戰情室 | 等待網址出現 |

> ⚠️ 僅供學術研究與教育用途，非投資建議，盈虧自負。


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║       🔑  STEP 1：填入所有 API 金鑰（先填完再往下執行）        ║
# ╚══════════════════════════════════════════════════════════════╝

import os

# ── [必填] ngrok 公開網址隧道（免費申請：https://dashboard.ngrok.com/）
NGROK_TOKEN    = ''  # 填入你的 ngrok authtoken

# ── [選填] Gemini AI 分析功能（https://aistudio.google.com/apikey）
GEMINI_API_KEY = ''  # 不填則 AI 評斷功能停用

# ── [選填] FinMind 台股資料（https://finmindtrade.com/ 免費版有限制）
FINMIND_TOKEN  = ''  # 不填則財報/法人資料降級使用備援

# ── 寫入環境變數 ────────────────────────────────────────────────
os.environ['NGROK_TOKEN']    = NGROK_TOKEN
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['FINMIND_TOKEN']  = FINMIND_TOKEN

print('✅ 金鑰設定完成')
if not NGROK_TOKEN:    print('⚠️  NGROK_TOKEN 未填，無法產生公開網址')
if not GEMINI_API_KEY: print('ℹ️  GEMINI_API_KEY 未填，AI 評斷功能停用')
if not FINMIND_TOKEN:  print('ℹ️  FINMIND_TOKEN 未填，部分台股資料降級備援')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  📦 STEP 2：安裝套件 + 從 GitHub 取得最新程式碼              ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, os

# ── 安裝所有依賴套件 ─────────────────────────────────────────────
print('📦 安裝套件中（約 2-3 分鐘）...')
!pip install -q streamlit pyngrok yfinance pandas plotly \
    google-generativeai FinMind nest-asyncio \
    beautifulsoup4 lxml html5lib ta requests
print('✅ 套件安裝完成')

# ── 從 GitHub 取得最新程式碼 ──────────────────────────────────────
REPO_DIR = '/content/my-stock-dashboard'
REPO_URL = 'https://github.com/Chenlin1984/my-stock-dashboard.git'

if os.path.exists(REPO_DIR):
    print('🔄 更新程式碼...')
    !cd {REPO_DIR} && git pull origin main
else:
    print('📥 下載程式碼...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'✅ 工作目錄：{os.getcwd()}')
!ls -la


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🔧 STEP 3：修復 Colab uvloop 與 asyncio 衝突                ║
# ╚══════════════════════════════════════════════════════════════╝

import asyncio, sys, os

# Colab 預設 uvloop，FinMind 的 nest_asyncio 無法 patch uvloop
# 必須在所有 import 之前強制切換回 DefaultEventLoopPolicy
asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

# 寫入 secrets.toml（Streamlit 讀取 API Key 的方式）
import os
NGROK_TOKEN    = os.environ.get('NGROK_TOKEN', '')
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
FINMIND_TOKEN  = os.environ.get('FINMIND_TOKEN', '')

os.makedirs('/content/my-stock-dashboard/.streamlit', exist_ok=True)
with open('/content/my-stock-dashboard/.streamlit/secrets.toml', 'w') as f:
    f.write(f'GEMINI_API_KEY = "{GEMINI_API_KEY}"\n')
    f.write(f'FINMIND_TOKEN  = "{FINMIND_TOKEN}"\n')

print('✅ 環境修復完成，secrets.toml 已寫入')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🔍 STEP 4（選擇性）：診斷資料源連線                         ║
# ╚══════════════════════════════════════════════════════════════╝

import yfinance as yf, requests, datetime

print('═'*50)
print('🧪 診斷測試開始')
print('═'*50)

# 1. yfinance 測試（0050.TW + SPY）
for ticker in ['0050.TW', 'SPY']:
    try:
        df = yf.Ticker(ticker).history(period='5d', auto_adjust=True)
        if not df.empty:
            print(f'✅ yfinance {ticker}: {len(df)} 筆，最新={df.index[-1].date()}，收盤={df["Close"].iloc[-1]:.2f}')
        else:
            print(f'⚠️  yfinance {ticker}: 空資料')
    except Exception as e:
        print(f'❌ yfinance {ticker}: {e}')

# 2. TWSE 公開資料測試
try:
    r = requests.get('https://www.twse.com.tw/rwd/zh/afterTrading/FMTQIK',
                     params={'response':'json','date':'20240101'}, timeout=10)
    if r.status_code == 200:
        print(f'✅ TWSE API: 連線正常 (HTTP {r.status_code})')
    else:
        print(f'⚠️  TWSE API: HTTP {r.status_code}')
except Exception as e:
    print(f'❌ TWSE API: {e}')

# 3. FinMind 測試
import os
fm_token = os.environ.get('FINMIND_TOKEN','')
if fm_token:
    try:
        r = requests.get('https://api.finmindtrade.com/api/v4/login',
                         params={'user_id':'','password':'','token':fm_token}, timeout=10)
        print(f'✅ FinMind Token: {"有效" if r.status_code==200 else "無效 HTTP "+str(r.status_code)}')
    except Exception as e:
        print(f'❌ FinMind: {e}')
else:
    print('ℹ️  FinMind Token 未設定，將使用備援資料源')

# 4. Gemini AI 測試
gkey = os.environ.get('GEMINI_API_KEY','')
if gkey:
    try:
        r = requests.post(
            f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent',
            params={'key': gkey},
            json={'contents': [{'parts': [{'text': 'hi'}]}]}, timeout=15)
        print(f'✅ Gemini API: {"正常" if r.status_code==200 else "HTTP "+str(r.status_code)}')
    except Exception as e:
        print(f'❌ Gemini: {e}')
else:
    print('ℹ️  Gemini API Key 未設定，AI 評斷功能停用')

print('═'*50)
print('🏁 診斷完成')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🚀 STEP 5：啟動台股 AI 戰情室                                ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, threading, time, sys, os, re, asyncio, signal

asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
REPO_DIR    = '/content/my-stock-dashboard'
os.chdir(REPO_DIR)

# ── Streamlit 設定（關閉 CORS / XSRF）──────────────────────────
os.makedirs(f'{REPO_DIR}/.streamlit', exist_ok=True)
with open(f'{REPO_DIR}/.streamlit/config.toml', 'w') as f:
    f.write('[server]\n')
    f.write('headless = true\n')
    f.write('enableCORS = false\n')
    f.write('enableXsrfProtection = false\n')
    f.write('port = 8501\n')

# ── 啟動 Streamlit ───────────────────────────────────────────────
log_file = '/tmp/streamlit.log'
with open(log_file, 'w') as lf:
    proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py',
         '--server.port', '8501', '--server.headless', 'true'],
        stdout=lf, stderr=lf,
        cwd=REPO_DIR
    )
print(f'✅ Streamlit PID: {proc.pid}')
time.sleep(4)

# ── 建立 ngrok 隧道 ──────────────────────────────────────────────
public_url = None
if NGROK_TOKEN:
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = NGROK_TOKEN
        tunnel = ngrok.connect(8501, proto='http')
        public_url = tunnel.public_url
        print(f'\n🎉 公開網址（ngrok）：{public_url}')
    except Exception as e:
        print(f'⚠️  ngrok 失敗：{e}')

if not public_url:
    # 備援：localhost proxy
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8501)
        print('🔗 已開啟 Colab 內建 proxy（僅限 Colab 內使用）')
    except Exception:
        print('ℹ️  請設定 NGROK_TOKEN 以取得公開網址')
        print('   或在瀏覽器開啟：http://localhost:8501')

print('\n💡 關閉服務：執行 proc.terminate() 或重啟 Runtime')
print('📋 查看日誌：!tail -50', log_file)


In [ ]:
# 🔍 查看啟動日誌（出現問題時執行）
!tail -80 /tmp/streamlit.log


In [ ]:
# 🛑 關閉 Streamlit（需要時執行）
try:
    proc.terminate()
    print('✅ Streamlit 已關閉')
except NameError:
    import subprocess
    subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
    print('✅ streamlit 程序已終止')

# 清除 ngrok 隧道
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('✅ ngrok 已關閉')
except Exception:
    pass
